# Scale up the Gauss-Newton for much large problems using the new construction 

See 'toshow' notebook for a demonstration of what I'm trying to rewrite in a more streamlined verison. 

In [28]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
from typing import Sequence
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import itertools
from jax.scipy.linalg import block_diag

# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


"## Module definition

We're doing just a simple MLP with *differing* dimensions, but also be able to store the global adjoints. 

In [2]:
def append_adjoints(adjoints):
    # print(adjoints)
    global_adjoints.append(adjoints)

class AdjointHook(nn.Module):
    """
    A wrapper module that applies a custom VJP to any given layer
    to capture its incoming gradient (adjoint) during the backward pass.
    """
    layer: nn.Module  # The layer to wrap

    @nn.compact
    def __call__(self, *args, **kwargs):
        # Define the function to which we'll attach the custom VJP
        @jax.custom_vjp
        def layer_with_hook(params, *args, **kwargs):
            return self.layer.apply(params, *args, **kwargs)

        def layer_fwd(params, *args, **kwargs):
            output = self.layer.apply(params, *args, **kwargs)
            return output, (params, args, kwargs)

        def layer_bwd(res, g):
            params, args, kwargs = res

            # Calculate the VJP of the original wrapped layer
            # This computes the gradients w.r.t. params and inputs
            _, vjp_fun = jax.vjp(
                lambda p, *a, **kw: self.layer.apply(p, *a, **kw), params, *args, **kwargs
            )
            
            # The VJP function returns a tuple of gradients
            grad_params, *grad_args = vjp_fun(g)

            # Need to store here
            global_adjoints.append(grad_args[0])
            # jax.debug.callback(append_adjoints, grad_args[0])

            return (grad_params,) + tuple(grad_args)

        # Attach the custom forward and backward functions
        layer_with_hook.defvjp(layer_fwd, layer_bwd)
        
        # Get the parameters for the wrapped layer
        layer_params = self.param('wrapped_layer', self.layer.init, *args, **kwargs)

        return layer_with_hook(layer_params, *args, **kwargs)


In [3]:
class MLP_Layer(nn.Module):
    num_units: int
    def setup(self):
        self.dense1 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
        self.dense2 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
    def __call__(self, x):
        f = self.dense1(x)
        f = nn.tanh(f)
        f = self.dense2(f)
        return f 

# Define the MLP module
class MLP(nn.Module):
    features: Sequence[int]
    
    def setup(self):
        layers = []
        for i, dim in enumerate(self.features):
            layers.append(
                AdjointHook(MLP_Layer(
                    num_units=dim
                ))
            )
        self.layers = tuple(layers)

    def __call__(self, x: jnp.ndarray):
        self.sow('intermediates', f'layer_{0}_output', x)
        for i, layer in enumerate(self.layers):
            x = self.perturb(f'layer_{i+1}', x)
            x = layer(x)
            self.sow('intermediates', f'layer_{i+1}_output', x)
        return x



In [4]:
n = 5
n_samples = 3

X = jax.random.normal(jax.random.PRNGKey(0), (n_samples, n))

key = jax.random.PRNGKey(0)
model = MLP(features=[4, 6, 4, 2])
params = model.init(key, jnp.ones((1, n)))
params, perturbations = params['params'], params['perturbations']
output, intermediates = model.apply(
    {'params': params, 'perturbation': perturbations}, X, mutable=['intermediates'])
intermediates

{'intermediates': {'layer_0_output': (Array([[-0.20584214, -0.78476578,  1.81608667,  0.18784401,  0.08086788],
          [-0.37211079,  1.19016372,  0.33864229,  0.08482584, -0.87181784],
          [ 1.05451609, -1.5594979 ,  0.36753958,  2.51635215,  0.25856516]],      dtype=float64),),
  'layer_1_output': (Array([[-0.89904447, -0.11885086, -0.10710438,  0.07907957],
          [ 0.94242466, -0.61541522, -0.34784817,  0.61104946],
          [-0.9057704 , -0.26950049,  1.05573896, -0.73687979]],      dtype=float64),),
  'layer_2_output': (Array([[ 0.08169137, -0.08571022,  0.25733173, -0.23046946,  0.38195217,
           -0.01852744],
          [ 0.22445845, -0.68290785, -0.5557547 , -0.06032243, -0.26355474,
            0.67336084],
          [ 0.17531121,  0.56170393, -0.43362155,  0.23902341,  0.31113846,
           -0.23976977]], dtype=float64),),
  'layer_3_output': (Array([[ 0.08793699,  0.18022425,  0.12992361,  0.06108001],
          [ 0.00729465, -0.53559186, -0.60916585,  0.01175448],
          [-0.18938253, -0.12906232, -0.121577  ,  0.17549349]],      dtype=float64),),
  'layer_4_output': (Array([[ 0.02831743, -0.02088565],
          [ 0.09441049,  0.06284346],
          [-0.06384981, -0.06087738]], dtype=float64),)}}

In [5]:
global_adjoints = []
jacobian = jax.jacobian(model.apply, argnums=0)({'params': params, 'perturbation': perturbations}, X)['params']
flattened, _ = jax.tree.flatten(
    jacobian
)

# Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
reshaped_leaves = [leaf.reshape(n_samples * model.features[-1], 1, -1) for leaf in flattened]
aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
J = aggregated_array.reshape((n_samples * model.features[-1], -1))

# treescope.display(global_adjoints[0].val)
# treescope.display(global_adjoints[1].val)
# treescope.display(global_adjoints[2].val)
# treescope.display(global_adjoints[3].val)
#
# # global_adjoints[0].val

In [33]:
# global_adjoints = []
jacobian = jax.jacobian(
    lambda params, perturbations, X: model.apply({'params': params, 'perturbations': perturbations}, X), argnums=1
)(params, perturbations, X)
flattened, _ = jax.tree.flatten(
    jacobian
)
together = [jnp.squeeze(leaf, axis=2).reshape((leaf.shape[0] * leaf.shape[1], -1)) for leaf in flattened]
my_adjoints = [
    block_diag(*jnp.split(leaf, n_samples, axis=0)) for leaf in together
]
my_adjoints.append(jnp.eye(J.shape[0]))
my_adjoints

# Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
# reshaped_leaves = [leaf.reshape(n_samples * model.features[-1], 1, -1) for leaf in flattened]
# aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
# J = aggregated_array.reshape((n_samples * model.features[-1], -1))
# treescope.display(J)
# treescope.display(global_adjoints)
# # global_adjoints[0].val

[Array([[-0.07109164,  0.0396372 ,  0.02243303,  0.01990836, -0.01441825,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [-0.00870956,  0.03645475,  0.01054312, -0.00413445, -0.01043811,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         -0.06384984,  0.02270973,  0.02589164,  0.00921017,  0.00588946,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         -0.00800875,  0.01736369,  0.00831618, -0.00604469, -0.00053625,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         -0.04571864,  0.0059279 ,  0.00264869,  0.02611178, -0.01585214],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         -0.00821517,  0.00668094, -0.00370484,  0.00702669, -0.01368862]],      dtype=float64),
 Array([[-0.0103804 , -0.08152029, -0.03375823,  0.07751273,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.02302214, -0.02829172, -0.03517704,  0.01471853,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        , -0.02976399,
         -0.06723424, -0.05516304,  0.05583244,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.01549351,
         -0.02202259, -0.04474154, -0.0017769 ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        , -0.007771  , -0.07596574,
         -0.00611007,  0.07783328],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.0374461 , -0.03513943,
         -0.01443885,  0.00073119]], dtype=float64),
 Array([[ 0.06883795, -0.05778381,  0.02117707, -0.06636137, -0.00508986,
          0.07041277,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ],
        [-0.10812159,  0.00256413, -0.04154576, -0.10711263, -0.06452832,
          0.06455857,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.07667069, -0.06359087,  0.01920951, -0.05560658,
         -0.00877541,  0.05408069,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        , -0.09138722, -0.00229623, -0.03733569, -0.0933203 ,
         -0.06010799,  0.05242888,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.06513269, -0.05566293,  0.0158822 ,
         -0.06762875, -0.0036507 ,  0.07256837],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.    

### Global adjoints

For now, we simply use the `jax.jacobian` rather than a clever thing; a bit larger memory footprint but the inversion is the time killer.

In [7]:
# I don't know how to store the adjoints in the model itself...
global_adjoints = []
global_adjoints = []
jacobian = jax.jacobian(
    lambda params, perturbations, X: model.apply({'params': params, 'perturbations': perturbations}, X), argnums=0
)(params, perturbations, X)

global_adjoints.reverse() # Backprop, so reverse to make more sense first
global_adjoints = [adj.val.reshape(adj.val.shape[0], -1) for adj in global_adjoints]
global_adjoints.append(jnp.eye(J.shape[0]))

In [8]:
global_adjoints

[Array([[-0.07109164,  0.0396372 ,  0.02243303,  0.01990836, -0.01441825,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [-0.00870956,  0.03645475,  0.01054312, -0.00413445, -0.01043811,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         -0.06384984,  0.02270973,  0.02589164,  0.00921017,  0.00588946,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         -0.00800875,  0.01736369,  0.00831618, -0.00604469, -0.00053625,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         -0.04571864,  0.0059279 ,  0.00264869,  0.02611178, -0.01585214],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         -0.00821517,  0.00668094, -0.00370484,  0.00702669, -0.01368862]],      dtype=float64),
 Array([[-0.0103804 , -0.08152029, -0.03375823,  0.07751273,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.02302214, -0.02829172, -0.03517704,  0.01471853,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        , -0.02976399,
         -0.06723424, -0.05516304,  0.05583244,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.01549351,
         -0.02202259, -0.04474154, -0.0017769 ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        , -0.007771  , -0.07596574,
         -0.00611007,  0.07783328],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.0374461 , -0.03513943,
         -0.01443885,  0.00073119]], dtype=float64),
 Array([[ 0.06883795, -0.05778381,  0.02117707, -0.06636137, -0.00508986,
          0.07041277,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ],
        [-0.10812159,  0.00256413, -0.04154576, -0.10711263, -0.06452832,
          0.06455857,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.07667069, -0.06359087,  0.01920951, -0.05560658,
         -0.00877541,  0.05408069,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        , -0.09138722, -0.00229623, -0.03733569, -0.0933203 ,
         -0.06010799,  0.05242888,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.06513269, -0.05566293,  0.0158822 ,
         -0.06762875, -0.0036507 ,  0.07256837],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
          0.    

In [9]:
global_adjoints[-1]

Array([[1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1.]], dtype=float64)

### Layer by Layer

Now need a good way of extracting the layer by layer

In [10]:
prediction, intermediates = model.apply({'params': params}, X, mutable=['intermediates'])

def get_layer_params(model: nn.Module, params: dict, layer_num: int, intermediates: jnp.ndarray):
    """
    Given the MLP model from above; extract out the layer, take the
    """
    params = {'params': params}
    layer = MLP_Layer(num_units=model.features[layer_num])
    def apply_layer(params):
        return layer.apply(
            params,
            intermediates['intermediates'][f'layer_{layer_num}_output'][0]
        )
    k_i = jax.jacrev(apply_layer)
    # treescope.display(k_i(params['params'][f'layers_{layer_num}']['wrapped_layer']))
    flattened, _ = jax.tree.flatten(
            k_i(params['params'][f'layers_{layer_num}']['wrapped_layer'])
    )

    reshaped_leaves = [leaf.reshape(leaf.shape[0] * leaf.shape[1], -1) for leaf in flattened]
    aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1) #.reshape(n * n_samples, -1)

    return aggregated_array

print(model.features)
get_layer_params(model, params, 2, intermediates)

[4, 6, 4, 2]


Array([[ 5.76904118e-01, -7.47030228e-02,  6.82531357e-01,
        -4.58656907e-01,  4.71280888e-02, -6.10259222e-03,
         5.57569191e-02, -3.74683104e-02, -4.94465791e-02,
         6.40281243e-03, -5.84999099e-02,  3.93115841e-02,
         1.48455739e-01, -1.92234572e-02,  1.75636962e-01,
        -1.18026972e-01, -1.32958785e-01,  1.72167644e-02,
        -1.57302633e-01,  1.05706409e-01,  2.20349789e-01,
        -2.85329800e-02,  2.60694325e-01, -1.75184995e-01,
        -1.06885573e-02,  1.38405582e-03, -1.26455585e-02,
         8.49773828e-03,  1.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  4.25337143e-02,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         2.85066329e-02,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.73151597e-01,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.17970973e-01,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-9.49316993e-02,  1.00928238e-02,  6.43710554e-01,
         6.29412532e-01, -7.75510073e-03,  8.24496616e-04,
         5.25855981e-02,  5.14175706e-02,  8.13661702e-03,
        -8.65058159e-04, -5.51725738e-02, -5.39470837e-02,
        -2.44289394e-02,  2.59720394e-03,  1.65647149e-01,
         1.61967814e-01,  2.18788590e-02, -2.32608779e-03,
        -1.48355633e-01, -1.45060360e-01, -3.62593681e-02,
         3.85497604e-03,  2.45866641e-01,  2.40405470e-01,
         1.75884145e-03, -1.86994206e-04, -1.19263092e-02,
        -1.16614029e-02,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         4.25337143e-02,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  2.85066329e-02,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.73151597e-01,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.17970973e-01,  0.00000000e+00,  0.00000000e+00],
       [-6.13076746e-01, -4.54085171e-01,  7.50037372e-01,
         2.90060580e-01, -5.00830822e-02, -3.70948426e-02,
         6.12715818e-02,  2.36954466e-02,  5.25469445e-02,
         3.89197394e-02, -6.42858669e-02, -2.48611569e-02,
        -1.57764107e-01, -1.16850525e-01,  1.93008408e-01,
         7.46417940e-02,  1.41295478e-01,  1.04652770e-01,
        -1.72860712e-01, -6.68501109e-02, -2.34165996e-01,
        -1.73438817e-01,  2.86478400e-01,  1.10789269e-01,
         1.13587435e-02,  8.41303635e-03, -1.38962734e-02,
        -5.37408050e-03,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  4.25337143e-02,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  2.85066329e-02,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.73151597e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.17970973e-01,  0.00000000e+00],
       [ 6.24001503e-01, -9.13851112e-02, -2.53489703e-01,
         7.42872179e-01,  5.09755388e-02, -7.46537466e-03,
        -2.07079221e-02,  6.06862456e-02, -5.34833074e-02,
         7.83263799e-03,  2.17266586e-02, -6.36717379e-02,
         1.60575390e-01, -2.35162880e-02, -6.52309433e-02,
         1.91164583e-01, -1.43813297e-01,  2.10614763e-02,
         5.84216379e-02, -1.71209350e-01,  2.38338724e-01,
        -3.49047408e-02, -9.68209505e-02,  2.83741653e-01,
        -1.15611507e-02,  1.69313222e-03,  4.69651585e-03,
        -1.37635199e-02,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  4.25337143e-02,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         2.85066329e-02,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.73151597e-01,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.17970973e-01],
       [ 5.36889374e-01, -7.47080743e-02,  5.73412895e-01,
        -4.10465956e-01,  1.20509356e-01, -1.67688597e-02,
         1.28707379e-01, -9.21325535e-02, -3.66645962e-01,
         5.10187335e-02, -3.91588181e-01,  

In [11]:
k_list = []
for l in range(len(model.features)):
    k_list.append(get_layer_params(model, params, l, intermediates))


In [12]:
k_list

[Array([[-3.09175044e-01,  8.04482281e-01, -5.05032778e-01,
         -1.46840632e-01,  6.36412501e-02, -1.65596351e-01,
          1.03957027e-01,  3.02259903e-02,  2.42629990e-01,
         -6.31330132e-01,  3.96332443e-01,  1.15235500e-01,
         -5.61488688e-01,  1.46100950e+00, -9.17183340e-01,
         -2.66675323e-01, -5.80766834e-02,  1.51117176e-01,
         -9.48673859e-02, -2.75831334e-02, -2.50023305e-02,
          6.50567785e-02, -4.08409312e-02, -1.18746907e-02,
          1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00, -6.97506964e-02,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00, -5.14907718e-01,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          4.56366450e-01,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  3.50768089e-01,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00],
        [ 1.10838008e+00, -1.83550552e-01, -1.13333106e-01,
         -1.96356624e-01, -2.28151321e-01,  3.77824381e-02,
          2.33287290e-02,  4.04184684e-02, -8.69818747e-01,
          1.44044191e-01,  8.89399424e-02,  1.54093966e-01,
          2.01291418e+00, -3.33343714e-01, -2.05822751e-01,
         -3.56600642e-01,  2.08202556e-01, -3.44788730e-02,
         -2.12889463e-02, -3.68844159e-02,  8.96323472e-02,
         -1.48433447e-02, -9.16500855e-03, -1.58789437e-02,
          0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00, -6.97506964e-02,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         -5.14907718e-01,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  4.56366450e-01,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  3.50768089e-01,
          0.00000000e+00,  0.00000000e+00],
        [-4.30252403e-01, -1.96875334e-01, -1.52973821e-02,
         -6.50354326e-01,  8.85640755e-02,  4.05252390e-02,
          3.14884586e-03,  1.33870333e-01,  3.37647349e-01,
          1.54501021e-01,  1.20048616e-02,  5.10375857e-01,
         -7.81375647e-01, -3.57542664e-01, -2.77813710e-02,
         -1.18109989e+00, -8.08203369e-02, -3.69818509e-02,
         -2.87352176e-03, -1.22165173e-01, -3.47936004e-02,
         -1.59208905e-02, -1.23706693e-03, -5.25927767e-02,
          0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         -6.97506964e-02,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00, -5.14907718e-01,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00,  4.56366450e-01,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          3.50768089e-01,  0.00000000e+00],
        [-4.70616668e-01,  1.85065478e-01, -1.66459918e-01,
          7.38875568e-01,  9.68727469e-02, -3.80942747e-02,
          3.42644639e-02, -1.52091727e-01,  3.69323850e-01,
         -1.45233050e-01,  1.30632043e-01, -5.79844296e-01,
         -8.54680657e-01,  3.36094946e-01, -3.02305639e-01,
          1.34186208e+00, -8.84025246e-02,  3.47634442e-02,
         -3.12684998e-02,  1.38793349e-01, -3.80577743e-02,
          1.49658537e-02, -1.34612611e-02,  5.97513020e-02,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00, -6.97506964e-02,  0.00000000e+00,
          0.00000000e+00,  0.00000000e+00, -5.14907718e-01,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
          4.56366450e-01,  0.00000000e+00,  0.00000000e+00,
          0.00000000e+00,  3.50768089e-01],
        [-2.85059601e-01,  3.82111967e-01, -6.36101961e-01,
         -1.48317456e-01,  1.06073752e-01, -1.42187983e-01,
          2.36700401e-01,  5.51905222e-02, -3.39267582e-01,
          4.54775810e-01, -7.57065475e-01, -1.76522046e-01,
         -9.65332314e-02,  1.29399270e-01, -2.15411022e-01,
         -5.02265617e-02, -2.41804197e-02,  3.24129686e-02,
         -5.39578833e-02, -1.25811528e-02,  2.48520032e-01,
         -3.33132029e-01,  5.54565012e-01,  

## Let's verify again that the Jacobian can be computed like this

In [13]:
print(model.features, X.shape)
J

[4, 6, 4, 2] (3, 5)


Array([[-0.10910033,  0.02760334,  0.00209504, ...,  0.        ,
        -0.04121125,  0.        ],
       [-0.03026762,  0.03336325, -0.01033248, ..., -0.05853946,
         0.        , -0.04121125],
       [-0.06256738,  0.00455466,  0.01788734, ...,  0.        ,
        -0.07201649,  0.        ],
       [-0.00840239,  0.0118679 , -0.00547721, ...,  0.07545931,
         0.        , -0.07201649],
       [-0.08234546,  0.00562527, -0.00037124, ...,  0.        ,
         0.0416849 ,  0.        ],
       [-0.03175643,  0.00954946, -0.0167537 , ..., -0.08092227,
         0.        ,  0.0416849 ]], dtype=float32)

In [20]:
# Size of (n \times k) \times (n \times k)
sum_mat = jnp.eye(J.shape[0])
lamb = 0.1

# For each layer
for i in range(len(model.features)):
    # Each dgdt item is (n \times k) \times (p_i) where p_i
    # is parameter of that layer
    middle = k_list[i] @ k_list[i].T
    # Adjoints is really scaling as n * k**2
    sum_mat += global_adjoints[i+1] @ middle @ global_adjoints[i+1].T / lamb
    # treescope.display(middle)
    treescope.display(global_adjoints[i+1])


print(
    jnp.linalg.norm(sum_mat - (jnp.eye(J.shape[0]) + J @ J.T/lamb))
)
# sum_mat

1.6499868696736753e-05


In [15]:
jnp.linalg.norm(sum_mat - (jnp.eye(J.shape[0]) + J @ J.T/lamb))


Array(1.64998687e-05, dtype=float64)

## Let's write a thing which given the two lists and the input gradient, will figure this out

In [ ]:
# @jax.jit
def get_correction(k_list, global_adjoints, grad, lamb: float=1e-1):
    """
    Assumes grad is a vector
    """
    p_list = [k.shape[1] for k in k_list]
    p_list.insert(0, 0)
    indices = list(itertools.accumulate(p_list))
    j_shape = global_adjoints[0].shape[0]

    # Use the SMW formulation; first do Jv
    aux_var = jnp.zeros(j_shape)
    for i in range(len(model.features)):
        aux_var += global_adjoints[i+1] @ (k_list[i] @ grad[indices[i]:indices[i+1]])

    # Calculate I + 1 / lamb JJ^T and get its inverse
    sum_mat = jnp.eye(j_shape)
    for i in range(len(model.features)):
        sum_mat += global_adjoints[i+1] @ (k_list[i] @ k_list[i].T) @ global_adjoints[i+1].T / lamb
    aux_var = jnp.linalg.solve(sum_mat, aux_var)

    # Finally do J^T v and scale
    out = jnp.concat([k_list[i].T @ global_adjoints[i+1].T @ aux_var for i in range(len(model.features))]) / (lamb ** 2)

    # Don't forget the 1/lambda I
    return -out + grad / lamb

grad = jnp.arange(J.shape[1],) * 2
my_correct = get_correction(k_list, global_adjoints, grad, lamb=1e1)
my_correct = get_correction(k_list, global_adjoints, grad, lamb=1e0)
my_correct = get_correction(k_list, global_adjoints, grad, lamb=1e-2)
jnp.linalg.norm(jnp.linalg.solve(jnp.eye(J.shape[1]) * 1e-2 + J.T @ J, grad) - my_correct) / (my_correct.T @ my_correct)

In [ ]:
type(get_correction)

## With the above now we need to solve some problems

Let's try to learn a simple functional regression problem;

In [ ]:
num_samples = 100
X = jax.random.uniform(jax.random.PRNGKey(0), (num_samples, 1), minval=0, maxval=1)
y = jnp.sin(2 * jnp.pi * X) + jnp.tanh(X)

plt.scatter(X, y)

In [ ]:
def train_model_sgd(X_train, y_train, model, params, learning_rate, num_epochs):
    # Define the optimizer
    optimizer = optax.sgd(learning_rate)
    state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=optimizer)

    @jax.jit
    def train_step(state, batch_X, batch_y):
        """
        We use full batch
        """
        def loss_fn(params):
            predictions = state.apply_fn(params, batch_X)
            # Mean Squared Error loss on only the first thing; the rest of the things don't matter
            loss = jnp.mean(jnp.square(predictions[:, 0] - batch_y[:, 0]))
            return loss, predictions

        grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
        (loss, predictions), grads = grad_fn(state.params)
        state = state.apply_gradients(grads=grads)
        return state, loss

    losses = []
    print(f"Starting training for {num_epochs} epochs...")
    for epoch in tqdm(range(num_epochs)):
        state, loss = train_step(state, X_train, y_train)
        losses.append(loss)

    return state, losses

num_epochs = 5_000

key = jax.random.PRNGKey(0)

model = MLP(features=[8, 8, 8, 1])
params = model.init(key, jnp.ones((1, 1)))

print(model.apply(params, X).shape)

# Train the model
trained_state_sgd, sgd_losses = train_model_sgd(
    X, y, model, params, learning_rate=2e-3, num_epochs=num_epochs
)

In [ ]:
plt.semilogy(sgd_losses)

In [ ]:
def train_model_precond(X_train, y_train, model, params, learning_rate, num_epochs):
    # Define the optimizer
    optimizer = optax.sgd(learning_rate)
    state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=optimizer)

    # @jax.jit
    def train_step(state, batch_X, batch_y):
        """
        We use full batch
        """
        def loss_fn(params):
            predictions = state.apply_fn(params, batch_X)
            # Mean Squared Error loss on only the first thing; the rest of the things don't matter
            loss = jnp.mean(jnp.square(predictions[:, 0] - batch_y[:, 0]))
            return loss, predictions

        grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
        (loss, predictions), grads = grad_fn(state.params)

        # Explicitly construct Jacobian
        global_adjoints = []
        jacobian = jax.jacobian(state.apply_fn, argnums=0)(params, X)
        jax.debug.print('{a}', a=len(global_adjoints))
        flattened, _ = jax.tree.flatten(jacobian)
        reshaped_leaves = [leaf.reshape(batch_X.shape[0], n, -1) for leaf in flattened]
        aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
        jacobian = aggregated_array.reshape((batch_X.shape[0] * batch_X.shape[1], -1))

        jtj = jacobian.T @ jacobian

        lm_mat = 1e-1 * jnp.eye(jtj.shape[0]) + jtj
        flattened, back = jax.flatten_util.ravel_pytree(grads)
        altered = jnp.linalg.solve(lm_mat, flattened) # Solve directly
        grads = back(altered) # Convert vector back to PyTree

        state = state.apply_gradients(grads=grads)
        return state, loss

    losses = []
    print(f"Starting training for {num_epochs} epochs...")
    for epoch in tqdm(range(num_epochs)):
        state, loss = train_step(state, X_train, y_train)
        if loss > 1e2:
            break
        losses.append(loss)

    return state, losses

num_epochs = 5_000
n=1
key = jax.random.PRNGKey(0)

model = MLP(features=[8, 8, 8, 1])
params = model.init(key, jnp.ones((1, 1)))

# Train the model
trained_state_precond, precond_losses = train_model_precond(
    X, y, model, params, learning_rate=3e-4, num_epochs=num_epochs
)

In [ ]:
predicted_y = model.apply(trained_state_precond.params, X)

# Create a figure with two subplots
plt.figure(figsize=(12, 5))

# First subplot for True vs. Predicted Data
plt.subplot(1, 2, 1)  # 2 rows, 1 column, 1st subplot
plt.scatter(X[:, 0], y[:, 0], marker='o', label='True Data', alpha=0.5)
plt.scatter(X[:, 0], predicted_y[:, 0], marker='x', color='red', label='Predicted Data', alpha=0.6)
plt.title('1D Regression Problem: True vs. Predicted Data')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True)

# Second subplot for Training Loss over Epochs
plt.subplot(1, 2, 2)  # 2 rows, 1 column, 2nd subplot
plt.semilogy(precond_losses, label='Precond')
plt.semilogy(sgd_losses, label='SGD')
plt.title('Training Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)

# Show the combined plot
plt.tight_layout()
plt.show()